# Data Cleaning and Preprocessing

### 1. 원본 데이터 로드

원본 CSV 파일을 로드하고 기본 정보 확인

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

# 1. 원본 데이터 로드
print("=" * 60)
print("1. 원본 데이터 로드")
print("=" * 60)
df = pd.read_csv("/Users/neo/.kaggle/fitness_tracker_dataset.csv")
print(f"원본 데이터 크기: {df.shape}")
print(f"\n데이터 컬럼: {df.columns.tolist()}")
print(f"\n첫 5행:\n{df.head()}")
print(f"\n데이터 타입:\n{df.dtypes}")

1. 원본 데이터 로드
원본 데이터 크기: (1000000, 12)

데이터 컬럼: ['user_id', 'date', 'steps', 'calories_burned', 'distance_km', 'active_minutes', 'sleep_hours', 'heart_rate_avg', 'workout_type', 'weather_conditions', 'location', 'mood']

첫 5행:
   user_id        date  steps  calories_burned  distance_km  active_minutes  \
0      468  2023-01-01   4530          2543.02        16.10             613   
1      879  2023-01-01  11613          1720.76         8.10             352   
2      152  2023-01-01  27335          1706.35         3.57             236   
3      311  2023-01-01  13459          2912.38         6.41            1329   
4      759  2023-01-01  15378          3344.51        17.88              52   

   sleep_hours  heart_rate_avg workout_type weather_conditions location  \
0          1.5             176      Walking              Clear     Park   
1          6.3             128      Cycling                Fog     Park   
2          6.7             134         Yoga               Snow     Park   

### 2. 결측치 확인 및 처리

workout_type 결측치를 'Unknown'으로 채우고, 날짜 형식 오류 행 제거

In [2]:
# 2. 결측치 확인 및 처리
print("\n" + "=" * 60)
print("2. 결측치 확인 및 처리")
print("=" * 60)
print(f"전체 결측치 현황:\n{df.isnull().sum()}")

# workout_type의 결측치를 'Unknown'으로 채우기
if 'workout_type' in df.columns:
    missing_workout = df['workout_type'].isna().sum()
    print(f"\nworkout_type 결측치: {missing_workout} ({missing_workout/len(df)*100:.2f}%)")
    df['workout_type'] = df['workout_type'].fillna('Unknown')
    print("✓ workout_type 결측치를 'Unknown'으로 처리")

# 날짜 형식 변환
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    missing_dates = df['date'].isna().sum()
    if missing_dates > 0:
        print(f"\n날짜 형식 오류: {missing_dates}개 행")
        df = df[df['date'].notna()].reset_index(drop=True)
        print(f"✓ 날짜 형식 오류 행 제거 (남은 데이터: {len(df)}개 행)")


2. 결측치 확인 및 처리
전체 결측치 현황:
user_id                    0
date                       0
steps                      0
calories_burned            0
distance_km                0
active_minutes             0
sleep_hours                0
heart_rate_avg             0
workout_type          143120
weather_conditions         0
location                   0
mood                       0
dtype: int64

workout_type 결측치: 143120 (14.31%)
✓ workout_type 결측치를 'Unknown'으로 처리


### 3. 범주형 컬럼 분석

각 범주형 컬럼의 고유값, 개수, 비율 분석

In [3]:
# 3. 범주형 컬럼 분석
print("\n" + "=" * 60)
print("3. 범주형(Categorical) 컬럼 분석")
print("=" * 60)

# 범주형 컬럼 식별
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\n범주형 컬럼: {categorical_cols}\n")

# 각 범주형 컬럼의 고유값과 비율 출력
for col in categorical_cols:
    print(f"\n{'─' * 50}")
    print(f"컬럼명: {col}")
    print(f"{'─' * 50}")
    
    value_counts = df[col].value_counts()
    total = len(df)
    
    print(f"고유값 개수: {df[col].nunique()}")
    print(f"\n고유값별 개수 및 비율:")
    for value, count in value_counts.items():
        ratio = (count / total) * 100
        print(f"  {value:20s} : {count:6d} ({ratio:6.2f}%)")
    
    if df[col].isna().sum() > 0:
        nan_count = df[col].isna().sum()
        nan_ratio = (nan_count / total) * 100
        print(f"  {'[NaN]':20s} : {nan_count:6d} ({nan_ratio:6.2f}%)")


3. 범주형(Categorical) 컬럼 분석

범주형 컬럼: ['workout_type', 'weather_conditions', 'location', 'mood']


──────────────────────────────────────────────────
컬럼명: workout_type
──────────────────────────────────────────────────
고유값 개수: 7

고유값별 개수 및 비율:
  Unknown              : 143120 ( 14.31%)
  Cycling              : 143115 ( 14.31%)
  Gym Workout          : 143108 ( 14.31%)
  Running              : 142774 ( 14.28%)
  Walking              : 142668 ( 14.27%)
  Swimming             : 142627 ( 14.26%)
  Yoga                 : 142588 ( 14.26%)

──────────────────────────────────────────────────
컬럼명: weather_conditions
──────────────────────────────────────────────────
고유값 개수: 4

고유값별 개수 및 비율:
  Fog                  : 250417 ( 25.04%)
  Rain                 : 250059 ( 25.01%)
  Clear                : 249807 ( 24.98%)
  Snow                 : 249717 ( 24.97%)

──────────────────────────────────────────────────
컬럼명: location
──────────────────────────────────────────────────
고유값 개수: 5

고유값별 개수 및 비율:
  

### 4. 시계열 데이터 정렬 및 최종 저장

user_id와 date로 정렬하여 시계열 형태로 변환

In [4]:
# 4. user_id 및 date로 시계열 데이터 정렬
print("\n" + "=" * 60)
print("4. 시계열 데이터 정렬 (user_id, date별 정렬)")
print("=" * 60)

# date를 datetime으로 변환 (이미 변환되었으나 확인)
df['date'] = pd.to_datetime(df['date'])

# user_id와 date로 정렬
df = df.sort_values(by=['user_id', 'date']).reset_index(drop=True)

print(f"✓ user_id 및 date로 정렬 완료")
print(f"\n정렬된 데이터 샘플:")
print(f"\nuser_id별 그룹 샘플:")
for user_id in df['user_id'].unique()[:3]:
    user_data = df[df['user_id'] == user_id][['user_id', 'date', 'steps', 'workout_type']].head(5)
    print(f"\nUser {user_id}:")
    print(user_data)

print(f"\n전체 데이터 통계:")
print(f"  총 행(rows): {len(df)}")
print(f"  총 사용자 수: {df['user_id'].nunique()}")
print(f"  사용자당 평균 기록 수: {len(df) / df['user_id'].nunique():.2f}")


4. 시계열 데이터 정렬 (user_id, date별 정렬)
✓ user_id 및 date로 정렬 완료

정렬된 데이터 샘플:

user_id별 그룹 샘플:

User 1:
   user_id       date  steps workout_type
0        1 2023-01-02  21559      Cycling
1        1 2023-01-03   1762      Cycling
2        1 2023-01-05  21471      Walking
3        1 2023-01-06   4195     Swimming
4        1 2023-01-08  29530      Cycling

User 2:
      user_id       date  steps workout_type
1018        2 2023-01-01    953  Gym Workout
1019        2 2023-01-01   3383      Unknown
1020        2 2023-01-02  14919     Swimming
1021        2 2023-01-03  24117      Walking
1022        2 2023-01-03   5506      Walking

User 3:
      user_id       date  steps workout_type
1992        3 2023-01-02  24726  Gym Workout
1993        3 2023-01-03   6030      Running
1994        3 2023-01-04   6719      Running
1995        3 2023-01-05  12588      Walking
1996        3 2023-01-06  29673      Running

전체 데이터 통계:
  총 행(rows): 1000000
  총 사용자 수: 999
  사용자당 평균 기록 수: 1001.00
✓ user_id 및 date로 정렬

#### 4-1. 데이터 품질 분석 (결측치, 중복, 시간 범위)

각 사용자별 시계열 데이터의 연속성 분석

In [5]:
# 데이터 품질 분석: 중복, 결측, 시간 범위
import numpy as np
import pandas as pd

# 1. 사용자별 기록 개수 분석
records_per_user = df.groupby('user_id').size()
print(f"사용자별 기록 개수 통계:")
print(f"  평균: {records_per_user.mean():.2f}")
print(f"  중앙값: {records_per_user.median():.0f}")
print(f"  최소: {records_per_user.min()}, 최대: {records_per_user.max()}")
print(f"  1000개 이상의 기록을 가진 사용자: {(records_per_user >= 1000).sum()} / {len(records_per_user)}")
print()

# 2. 중복된 (user_id, date) 조합 확인
duplicates = df.groupby(['user_id', 'date']).size()
duplicated_count = (duplicates > 1).sum()
print(f"중복된 (user_id, date) 조합: {duplicated_count} 개")
if duplicated_count > 0:
    print(f"  총 중복 기록: {duplicates[duplicates > 1].sum()} 개")
    print(f"  예시:")
    dup_example = df[df.duplicated(subset=['user_id', 'date'], keep=False)].sort_values(['user_id', 'date']).head(10)
    print(dup_example[['user_id', 'date', 'steps', 'calories_burned']])
print()

# 3. 시간 범위 분석
date_range = pd.to_datetime(df['date'])
print(f"시간 범위:")
print(f"  시작: {date_range.min()}")
print(f"  종료: {date_range.max()}")
print(f"  일수: {(date_range.max() - date_range.min()).days} 일")
print()

# 4. 사용자별 시간 커버리지 분석
def analyze_user_coverage(user_df):
    date_min = pd.to_datetime(user_df['date']).min()
    date_max = pd.to_datetime(user_df['date']).max()
    date_range = pd.date_range(date_min, date_max, freq='D')
    coverage = len(user_df) / len(date_range)
    return len(user_df), len(date_range), coverage

user_stats = df.groupby('user_id').apply(
    lambda x: pd.Series({
        'num_records': analyze_user_coverage(x)[0],
        'expected_records': analyze_user_coverage(x)[1],
        'coverage': analyze_user_coverage(x)[2]
    })
)

print(f"사용자별 시간 커버리지 통계:")
print(f"  평균 커버리지: {user_stats['coverage'].mean():.2%}")
print(f"  중앙값 커버리지: {user_stats['coverage'].median():.2%}")
print(f"  최소 커버리지: {user_stats['coverage'].min():.2%}")
print(f"  80% 이상 커버리지: {(user_stats['coverage'] >= 0.8).sum()} 사용자")
print(f"  50% 이상 커버리지: {(user_stats['coverage'] >= 0.5).sum()} 사용자")
print()

# 5. 결론
print(f"결론:")
print(f"  - 완전 시계열 (1000개 레코드): {(records_per_user >= 1000).sum()} / 999 사용자 ({(records_per_user >= 1000).sum() / len(records_per_user) * 100:.1f}%)")
print(f"  - 중복 레코드 존재: {'예' if duplicated_count > 0 else '아니오'}")
print(f"  - 평균 커버리지: {user_stats['coverage'].mean():.2%}")
print(f"  - 권장사항: 중복 제거 후 빈 날짜 보간 필요")

사용자별 기록 개수 통계:
  평균: 1001.00
  중앙값: 1002
  최소: 900, 최대: 1094
  1000개 이상의 기록을 가진 사용자: 525 / 999

중복된 (user_id, date) 조합: 263927 개
  총 중복 기록: 631552 개
  예시:
    user_id       date  steps  calories_burned
4         1 2023-01-08  29530          1988.14
5         1 2023-01-08  18617          2792.98
6         1 2023-01-09   8921          2361.44
7         1 2023-01-09  10143          3906.60
8         1 2023-01-09  10122          2622.55
9         1 2023-01-09  10587          2503.75
16        1 2023-01-16  28209          2703.60
17        1 2023-01-16  14097          2106.15
18        1 2023-01-16   9305          2633.17
19        1 2023-01-19   9247          2248.47

시간 범위:
  시작: 2023-01-01 00:00:00
  종료: 2025-09-26 00:00:00
  일수: 999 일

    user_id       date  steps  calories_burned
4         1 2023-01-08  29530          1988.14
5         1 2023-01-08  18617          2792.98
6         1 2023-01-09   8921          2361.44
7         1 2023-01-09  10143          3906.60
8         1 2023-01-

### 4-2. 데이터 정제: 중복 제거 및 마스킹

중복은 제거하고, 결측 시간 단계는 0으로 채운 후 마스크 생성

In [42]:
# 중복 제거 및 마스킹 기반 처리
from datetime import datetime, timedelta

# 1. 중복 제거: (user_id, date) 당 하나의 레코드만 유지
# 중복이 있으면 첫 번째 기록만 선택 (집계 대신)
df_deduplicated = df.copy()
df_deduplicated['date'] = pd.to_datetime(df_deduplicated['date'])

# 연속형 컬럼과 범주형 컬럼 구분
continuous_cols = ['steps', 'calories_burned', 'distance_km', 'active_minutes', 'sleep_hours', 'heart_rate_avg']
categorical_cols = ['workout_type', 'weather_conditions', 'location', 'mood']

# 중복된 레코드 제거: (user_id, date)별로 첫 번째만 유지
df_deduplicated = df_deduplicated.drop_duplicates(subset=['user_id', 'date'], keep='first')

print(f"중복 제거 후 레코드 수: {len(df_deduplicated)}")
print(f"중복으로 제거된 레코드: {len(df) - len(df_deduplicated)}")
print()

# 2. 전체 시간 범위 결정
date_min = df_deduplicated['date'].min()
date_max = df_deduplicated['date'].max()
date_range = pd.date_range(date_min, date_max, freq='D')
print(f"전체 날짜 범위: {date_min} ~ {date_max} ({len(date_range)} 일)")
print()

# 3. 각 사용자마다 완전한 시계열 생성 (마스킹 기반)
user_ids = df_deduplicated['user_id'].unique()
dfs_masked = []

for user_id in user_ids:
    user_data = df_deduplicated[df_deduplicated['user_id'] == user_id].copy()
    
    # 해당 사용자의 시간 범위
    user_min_date = user_data['date'].min()
    user_max_date = user_data['date'].max()
    user_date_range = pd.date_range(user_min_date, user_max_date, freq='D')
    
    # 완전한 시간 시리즈 생성
    full_dates = pd.DataFrame({'date': user_date_range, 'user_id': user_id})
    
    # Merge: 기존 데이터와 결합 (left join으로 모든 날짜 포함)
    user_masked = full_dates.merge(user_data[['user_id', 'date'] + continuous_cols + categorical_cols], 
                                    on=['user_id', 'date'], how='left')
    
    # 결측값을 0으로 채우기 (연속형: 활동 없음, 범주형: 결측 표시)
    for col in continuous_cols:
        user_masked[col] = user_masked[col].fillna(0.0)
    
    for col in categorical_cols:
        user_masked[col] = user_masked[col].fillna('None')
    
    # 마스크 생성: 0 = 실제 데이터 있음, 1 = 결측 (활동 없음)
    user_masked['mask'] = (~user_masked['date'].isin(user_data['date'])).astype(float)
    
    dfs_masked.append(user_masked)

df_complete = pd.concat(dfs_masked, ignore_index=True)
df_complete = df_complete.sort_values(['user_id', 'date']).reset_index(drop=True)

# 마스크 열 확인
mask_coverage = df_complete.groupby('user_id')['mask'].agg(['sum', 'count', 'mean'])

print(f"마스킹 처리 후 레코드 수: {len(df_complete)}")
print(f"마스킹 통계 (사용자별 유효 데이터 비율):")
print(f"  평균: {mask_coverage['mean'].mean():.2%}")
print(f"  중앙값: {mask_coverage['mean'].median():.2%}")
print(f"  최소: {mask_coverage['mean'].min():.2%}")
print()

# 4. 정확히 1000개씩 선택
df_final = []
for user_id in user_ids:
    user_data = df_complete[df_complete['user_id'] == user_id]
    if len(user_data) >= 1000:
        user_data = user_data.iloc[:1000]  # 처음 1000개 선택
    df_final.append(user_data)

df_final = pd.concat(df_final, ignore_index=True)
print(f"최종 레코드 수: {len(df_final)}")
print(f"최종 사용자 수: {df_final['user_id'].nunique()}")
print(f"사용자별 레코드 수 (최종):")
print(df_final.groupby('user_id').size().describe())

중복 제거 후 레코드 수: 632375
중복으로 제거된 레코드: 367625

전체 날짜 범위: 2023-01-01 00:00:00 ~ 2025-09-26 00:00:00 (1000 일)

마스킹 처리 후 레코드 수: 997817
마스킹 통계 (사용자별 유효 데이터 비율):
  평균: 36.62%
  중앙값: 36.60%
  최소: 31.73%

마스킹 처리 후 레코드 수: 997817
마스킹 통계 (사용자별 유효 데이터 비율):
  평균: 36.62%
  중앙값: 36.60%
  최소: 31.73%

최종 레코드 수: 997817
최종 사용자 수: 999
사용자별 레코드 수 (최종):
count     999.000000
mean      998.815816
std         1.398579
min       991.000000
25%       998.000000
50%       999.000000
75%      1000.000000
max      1000.000000
dtype: float64
최종 레코드 수: 997817
최종 사용자 수: 999
사용자별 레코드 수 (최종):
count     999.000000
mean      998.815816
std         1.398579
min       991.000000
25%       998.000000
50%       999.000000
75%      1000.000000
max      1000.000000
dtype: float64


### 4-3. 텐서 생성 및 마스크와 함께 저장

특성 텐서와 마스크를 .npz 형식으로 저장

In [43]:
# 텐서 생성 및 마스크와 함께 저장
import numpy as np
from sklearn.preprocessing import LabelEncoder

# 1. 정제된 CSV 저장
output_csv_path = './fitness_tracker_dataset_refined.csv'
df_final.to_csv(output_csv_path, index=False)
print(f"정제된 데이터 저장: {output_csv_path}")
print(f"  크기: {len(df_final)} 레코드")
print()

# 2. 특성 벡터 추출 (10개 특성)
features = ['steps', 'calories_burned', 'distance_km', 'active_minutes', 'sleep_hours', 
            'heart_rate_avg', 'workout_type', 'weather_conditions', 'location', 'mood']

# 범주형 변수 인코딩
df_tensor = df_final.copy()
le_dict = {}

for cat_col in ['workout_type', 'weather_conditions', 'location', 'mood']:
    le = LabelEncoder()
    
    # 'None'을 포함한 모든 값 인코딩 (None = 0)
    all_classes = df_tensor[cat_col].unique()
    le.fit(np.sort(all_classes))
    
    df_tensor[cat_col] = le.transform(df_tensor[cat_col])
    le_dict[cat_col] = le
    print(f"{cat_col} 인코딩: {len(le.classes_)} 개 클래스 (None 포함)")
    print(f"  클래스: {le.classes_}")

print()

# 3. 999 × 1000 × 10 특성 텐서 생성
tensor_data = []
mask_data = []

for user_id in sorted(df_tensor['user_id'].unique()):
    user_df = df_tensor[df_tensor['user_id'] == user_id].copy()
    
    # 특성 추출
    user_features = user_df[features].values
    user_mask = user_df['mask'].values.reshape(-1, 1)
    
    # 길이 조정 (정확히 1000)
    if len(user_features) == 1000:
        tensor_data.append(user_features)
        mask_data.append(user_mask)
    elif len(user_features) > 1000:
        tensor_data.append(user_features[:1000])
        mask_data.append(user_mask[:1000])
    else:
        # 부족한 경우: 0으로 패딩하고 마스크는 0
        padding = np.zeros((1000 - len(user_features), len(features)))
        tensor_data.append(np.vstack([user_features, padding]))
        
        padding_mask = np.zeros((1000 - len(user_mask), 1))
        mask_data.append(np.vstack([user_mask, padding_mask]))

tensor = np.array(tensor_data, dtype=np.float32)
mask = np.array(mask_data, dtype=np.float32)

print(f"특성 텐서 형태: {tensor.shape}")
print(f"마스크 형태: {mask.shape}")
print()

# 4. .npz 형식으로 함께 저장
output_npz_path = './fitness_tracker_data.npz'
np.savez(output_npz_path, features=tensor, mask=mask)
print(f"✓ 텐서 저장: {output_npz_path}")
print()

# 5. 기본 통계
print(f"특성 텐서 통계:")
print(f"  최소값: {tensor.min():.2f}")
print(f"  최대값: {tensor.max():.2f}")
print(f"  평균: {tensor.mean():.2f}")
print(f"  표준편차: {tensor.std():.2f}")
print()

print(f"마스크 통계:")
print(f"  결측 데이터 비율 (mask=1): {mask.mean():.2%}")
print(f"  유효 데이터 비율 (mask=0): {(1 - mask.mean()):.2%}")
print()

print(f"✓ 데이터 정제 완료!")
print(f"  - 중복 제거: {len(df) - len(df_deduplicated):,} 레코드 (첫 번째만 유지)")
print(f"  - 마스킹 처리: {len(df_complete):,} 레코드 (완전 시계열)")
print(f"  - 최종 형태: 999 사용자 × 1,000 시간 단계 × (10 특성 + 1 마스크)")
print(f"  - 저장 방식: .npz (특성과 마스크 함께 저장)")
print(f"\n특성 설명:")
print(f"  연속형: steps, calories_burned, distance_km, active_minutes, sleep_hours, heart_rate_avg")
print(f"  범주형: workout_type(0-{len(le_dict['workout_type'].classes_)-1}), weather_conditions(0-{len(le_dict['weather_conditions'].classes_)-1}), location(0-{len(le_dict['location'].classes_)-1}), mood(0-{len(le_dict['mood'].classes_)-1})")
print(f"  마스크: 1.0=유효, 0.0=결측")

정제된 데이터 저장: ./fitness_tracker_dataset_refined.csv
  크기: 997817 레코드

workout_type 인코딩: 8 개 클래스 (None 포함)
  클래스: ['Cycling' 'Gym Workout' 'None' 'Running' 'Swimming' 'Unknown' 'Walking'
 'Yoga']
weather_conditions 인코딩: 5 개 클래스 (None 포함)
  클래스: ['Clear' 'Fog' 'None' 'Rain' 'Snow']
location 인코딩: 6 개 클래스 (None 포함)
  클래스: ['Gym' 'Home' 'None' 'Office' 'Other' 'Park']
mood 인코딩: 5 개 클래스 (None 포함)
  클래스: ['Happy' 'Neutral' 'None' 'Stressed' 'Tired']

location 인코딩: 6 개 클래스 (None 포함)
  클래스: ['Gym' 'Home' 'None' 'Office' 'Other' 'Park']
mood 인코딩: 5 개 클래스 (None 포함)
  클래스: ['Happy' 'Neutral' 'None' 'Stressed' 'Tired']

특성 텐서 형태: (999, 1000, 10)
마스크 형태: (999, 1000, 1)

✓ 텐서 저장: ./fitness_tracker_data.npz

특성 텐서 통계:
  최소값: 0.00
  최대값: 29999.00
  평균: 1178.71
  표준편차: 4260.49

마스크 통계:
  유효 데이터 비율 (mask=1): 36.58%
  결측 데이터 비율 (mask=0): 63.42%

✓ 데이터 정제 완료!
  - 중복 제거: 367,625 레코드 (첫 번째만 유지)
  - 마스킹 처리: 997,817 레코드 (완전 시계열)
  - 최종 형태: 999 사용자 × 1,000 시간 단계 × (10 특성 + 1 마스크)
  - 저장 방식: .npz (특성과 마스크 함께 저장)



# 5. File Check

In [44]:
# 확인: 파일이 저장되었는지 확인
import os

print("저장된 파일 목록:")
for f in ['fitness_tracker_dataset_refined.csv', 'fitness_tracker_data.npz']:
    path = f'./{f}'
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)  # MB
        print(f"  ✓ {f} ({size:.2f} MB)")
    else:
        print(f"  ✗ {f} (파일 없음)")

print("\n데이터 정제 파이프라인 완료!")
print(f"  입력: 1,000,000 레코드")
print(f"  처리:")
print(f"    - 중복 제거: 제거")
print(f"    - 결측 처리: 마스킹 기반 (0 채우기 + 마스크)")
print(f"  출력: 999 × 1000 × 10 특성 + 999 × 1000 × 1 마스크")
print(f"  준비: training.py와 GRU 모델 학습 준비 완료 ✓")

저장된 파일 목록:
  ✓ fitness_tracker_dataset_refined.csv (71.26 MB)
  ✓ fitness_tracker_data.npz (41.92 MB)

데이터 정제 파이프라인 완료!
  입력: 1,000,000 레코드
  처리:
    - 중복 제거: 제거
    - 결측 처리: 마스킹 기반 (0 채우기 + 마스크)
  출력: 999 × 1000 × 10 특성 + 999 × 1000 × 1 마스크
  준비: training.py와 GRU 모델 학습 준비 완료 ✓


# 6. Decoding Function

In [46]:
# 범주형 인코딩 맵 (디코딩용)
CATEGORICAL_ENCODINGS = {
    'workout_type': ['Cycling', 'Gym Workout', 'None', 'Running', 'Swimming', 'Unknown', 'Walking', 'Yoga'],
    'weather_conditions': ['Clear', 'Fog', 'None', 'Rain', 'Snow'],
    'location': ['Gym', 'Home', 'None', 'Office', 'Other', 'Park'],
    'mood': ['Happy', 'Neutral', 'None', 'Stressed', 'Tired']
}

FEATURE_ORDER = ['steps', 'calories_burned', 'distance_km', 'active_minutes', 'sleep_hours',
                 'heart_rate_avg', 'workout_type', 'weather_conditions', 'location', 'mood']

CONTINUOUS_FEATURES = ['steps', 'calories_burned', 'distance_km', 'active_minutes', 'sleep_hours', 'heart_rate_avg']
CATEGORICAL_FEATURES = ['workout_type', 'weather_conditions', 'location', 'mood']

def decode_sample(user_idx, time_idx):
    """
    특정 사용자, 특정 시점의 벡터를 디코딩
    
    Args:
        user_idx: 사용자 인덱스 (0-998)
        time_idx: 시점 인덱스 (0-999)
    """
    if user_idx < 0 or user_idx >= len(tensor):
        print(f"❌ 사용자 인덱스 범위 초과: {user_idx} (0-{len(tensor)-1})")
        return
    
    if time_idx < 0 or time_idx >= len(tensor[0]):
        print(f"❌ 시점 인덱스 범위 초과: {time_idx} (0-{len(tensor[0])-1})")
        return
    
    # 마스크 확인: 0.0 = 유효한 데이터, 1.0 = 결측값
    is_valid = mask[user_idx, time_idx, 0] == 0.0
    
    if is_valid == False:
        print(f"🚫 결측값입니다!")
        print(f"   사용자: {user_idx}, 시점: {time_idx}")
        print(f"   (활동이 없던 날짜입니다)")
        return
    
    # 특성 벡터
    feature_vector = tensor[user_idx, time_idx]
    
    print(f"\n✓ 사용자 {user_idx}, 시점 {time_idx} 데이터:")
    print(f"{'='*60}")
    
    # 연속형 변수
    print(f"\n【연속형 변수】")
    for i, feat_name in enumerate(CONTINUOUS_FEATURES):
        value = feature_vector[i]
        print(f"  {feat_name:20s}: {value:10.2f}")
    
    # 범주형 변수
    print(f"\n【범주형 변수】")
    categorical_start_idx = len(CONTINUOUS_FEATURES)
    for i, feat_name in enumerate(CATEGORICAL_FEATURES):
        encoded_value = int(feature_vector[categorical_start_idx + i])
        class_map = CATEGORICAL_ENCODINGS[feat_name]
        
        # 안전한 인덱스 처리
        if encoded_value < len(class_map):
            decoded_value = class_map[encoded_value]
        else:
            decoded_value = f"Unknown(#{encoded_value})"
        
        print(f"  {feat_name:20s}: {decoded_value} (인코딩값: {encoded_value})")
    
    print(f"\n【마스크】")
    print(f"  유효 데이터: ✓ (mask=0.0)")

# 사용 예시
print("사용 방법:")
print("  decode_sample(user_idx, time_idx)")
print("\n예시:")
decode_sample(0, 0)  # 첫 번째 사용자, 첫 번째 시점


사용 방법:
  decode_sample(user_idx, time_idx)

예시:

✓ 사용자 0, 시점 0 데이터:

【연속형 변수】
  steps               :   21559.00
  calories_burned     :    3045.06
  distance_km         :       0.32
  active_minutes      :     841.00
  sleep_hours         :      11.50
  heart_rate_avg      :     126.00

【범주형 변수】
  workout_type        : Cycling (인코딩값: 0)
  weather_conditions  : Rain (인코딩값: 3)
  location            : Park (인코딩값: 5)
  mood                : Stressed (인코딩값: 3)

【마스크】
  유효 데이터: ✓ (mask=0.0)


In [57]:
# 결측값 예시 테스트
print("\n" + "="*60)
print("결측값 예시 (마스크=0인 시점):")
print("="*60)

# 첫 번째 사용자에서 마스크가 1인 시점 찾기
user_0_mask = mask[0, :, 0]
missing_indices = np.where(user_0_mask == 1.0)[0]

if len(missing_indices) > 0:
    print(f"\n사용자 0의 결측 시점: {missing_indices[:5].tolist()} 찾음")
    decode_sample(0, missing_indices[0])
else:
    print(f"\n사용자 0은 모든 시점이 유효합니다.")



결측값 예시 (마스크=0인 시점):

사용자 0의 결측 시점: [2, 5, 15, 16, 19] 찾음
🚫 결측값입니다!
   사용자: 0, 시점: 2
   (활동이 없던 날짜입니다)


In [52]:
decode_sample(0, 100)


✓ 사용자 0, 시점 100 데이터:

【연속형 변수】
  steps               :   19426.00
  calories_burned     :    3238.19
  distance_km         :      12.69
  active_minutes      :     206.00
  sleep_hours         :       6.90
  heart_rate_avg      :     142.00

【범주형 변수】
  workout_type        : Swimming (인코딩값: 4)
  weather_conditions  : Rain (인코딩값: 3)
  location            : Home (인코딩값: 1)
  mood                : Tired (인코딩값: 4)

【마스크】
  유효 데이터: ✓ (mask=0.0)
